In [1]:
# core libs - chỉ cài nếu chưa có
import importlib.util
import subprocess
import sys

def install_if_missing(package_name, pip_name=None):
    """Cài đặt package nếu chưa có"""
    if pip_name is None:
        pip_name = package_name
    
    # Kiểm tra package đã cài chưa
    if importlib.util.find_spec(package_name) is None:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
    else:
        print(f"{package_name} already installed, skipping...")

packages = [
    ("fitz", "PyMuPDF==1.23.8"),
    ("pdf2image", "pdf2image"),
    ("cv2", "opencv-python-headless"),
    ("numpy", "numpy"),
    ("tqdm", "tqdm"),
    ("requests", "requests"),
    ("paddleocr", "paddleocr"),
    ("pytesseract", "pytesseract"),
    ("paddle", "paddlepaddle"),
]

for package, pip_name in packages:
    install_if_missing(package, pip_name)

print("\nAll packages ready!")

fitz already installed, skipping...
pdf2image already installed, skipping...
cv2 already installed, skipping...
numpy already installed, skipping...
tqdm already installed, skipping...
requests already installed, skipping...
paddleocr already installed, skipping...
pytesseract already installed, skipping...
paddle already installed, skipping...

All packages ready!


# Libraries

In [2]:
import requests
import fitz  # PyMuPDF
import io
import os
import json
import re
from tqdm.auto import tqdm
import numpy as np
import pytesseract
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'



import cv2
import matplotlib.pyplot as plt
from PIL import Image

# Config

In [19]:
DPI = 200   
FIGURE_MIN_AREA = 2000  
GEOMETRY_KEYWORDS = [
    # Từ khóa cũ
    "tam giác", "đường tròn", "đoạn thẳng", "góc", "vuông góc", "song song", "trung điểm",
    "tiếp tuyến", "bán kính", "hình chữ nhật", "hình vuông", "hình thang", "chu vi",
    "diện tích", "đỉnh", "chân", "đường cao", "phân giác", "điểm", "giao điểm", "tâm",
    
    # Từ khóa mới - Từ mục lục
    "hình","hình chóp", "tứ giác", "pythagore", "pitago", "định lí", "định lý",
    "hình bình hành", "hình thoi", "hình vuông", "hình chữ nhật",
    "thể tích", "xung quanh", "đáy", "cạnh bên",
    
    # Tiếng Anh
    "triangle", "circle", "rectangle", "square", "parallelogram", "trapezoid",
    "pythagoras", "theorem", "volume", "area", "perimeter",
    
    # Viết tắt và ký hiệu
    "abc", "abcd", "∆", "∠", "⊥", "//",
]

In [10]:
PDF_URL = "https://8486fef5bc.vws.vegacdn.vn/data/doc/2025/thcschuvananq1/2025_2/3/sach-giao-khoa-toan-8-tap-1-chan-troi-sang-tao_3220251.pdf"
OUTPUT_JSON = "geometry_extracted.json"

response = requests.get(PDF_URL)
pdf_bytes = response.content

In [14]:
PDF_PATH = "D:\\Nhi\\VS_code\\capstone_draft\\dataset\\sach-bai-tap-toan-7-tap-1-ket-noi-tri-thuc-voi-cuoc-song.pdf"
OUTPUT_JSON = "geometry_extracted_7.json"

with open(PDF_PATH, 'rb') as f:
    pdf_bytes = f.read()

In [20]:
doc = fitz.open("pdf", pdf_bytes)
print(f"PDF loaded. Total pages: {len(doc)}")

PDF loaded. Total pages: 138


# Extract text from PDF

In [21]:
def pdf_to_images(doc, dpi=200):
    images = []
    for i, page in enumerate(doc):
        pix = page.get_pixmap(dpi=dpi)
        img = Image.open(io.BytesIO(pix.tobytes("png")))
        images.append((i, img))
    return images

images = pdf_to_images(doc, dpi=200)
print(f"Total pages converted to images: {len(images)}")

Total pages converted to images: 138


In [22]:
geometry_pages_dir = "geometry_pages_7"
os.makedirs(geometry_pages_dir, exist_ok=True)

In [23]:
geometry_pages = []
geometry_results = []

num_pages_to_check = len(images) 
print(f"Checking {num_pages_to_check} pages\n")

for idx in tqdm(range(num_pages_to_check), desc="Processing pages"):
    page_num, img = images[idx]
    
    # Method 1: Extract text directly from PDF
    page = doc[page_num]
    text_pdf = page.get_text().lower()
    
    # Method 2: OCR to extract text from image
    try:
        text_ocr = pytesseract.image_to_string(img, lang='vie+eng').lower()
    except:
        text_ocr = ""
    
    # Combine both methods
    text = text_pdf + " " + text_ocr
    
    # Check for geometry keywords
    if any(keyword in text for keyword in GEOMETRY_KEYWORDS):
        geometry_pages.append(page_num)
        
        # Save page image
        img_path = os.path.join(geometry_pages_dir, f"page_{page_num+1}.png")
        img.save(img_path)
        
        # Save page information
        matched_kw = [kw for kw in GEOMETRY_KEYWORDS if kw in text]
        geometry_results.append({
            "page": page_num + 1,
            "image_path": img_path,
            "detected_text_pdf": text_pdf[:250],
            "detected_text_ocr": text_ocr[:250],
            "matched_keywords": matched_kw
        })
        
        print(f"\nPage {page_num+1}: FOUND - {matched_kw[:3]}...")

print(f"RESULTS: Found {len(geometry_pages)} geometry pages")
print(f"Pages: {[p+1 for p in geometry_pages]}")

if geometry_pages:
    print(f"\nImages saved at: {os.path.abspath(geometry_pages_dir)}")
    
    # Save JSON file
    detail_json = os.path.join(geometry_pages_dir, "detail.json")
    with open(detail_json, "w", encoding="utf-8") as f:
        json.dump(geometry_results, f, ensure_ascii=False, indent=2)
    print(f"Details saved at: {detail_json}")
else:
    print(f"\nNo geometry pages found in {num_pages_to_check} pages")


Checking 138 pages



Processing pages:   0%|          | 0/138 [00:00<?, ?it/s]


Page 5: FOUND - ['song song']...

Page 39: FOUND - ['song song', '//']...

Page 40: FOUND - ['song song', 'abc', '//']...

Page 41: FOUND - ['//']...

Page 42: FOUND - ['song song', '//']...

Page 43: FOUND - ['song song', '//']...

Page 44: FOUND - ['//']...

Page 45: FOUND - ['//']...

Page 46: FOUND - ['song song', '//']...

Page 47: FOUND - ['song song']...

Page 49: FOUND - ['song song']...

Page 50: FOUND - ['//']...

Page 51: FOUND - ['song song', 'abc', '//']...

Page 52: FOUND - ['abc']...

Page 54: FOUND - ['abc']...

Page 55: FOUND - ['abc']...

Page 56: FOUND - ['abc']...

Page 57: FOUND - ['abc']...

Page 58: FOUND - ['abc', 'abcd']...

Page 59: FOUND - ['abc', 'abcd']...

Page 60: FOUND - ['abc']...

Page 61: FOUND - ['abc']...

Page 62: FOUND - ['song song', 'abc']...

Page 63: FOUND - ['song song', 'abc', 'abcd']...

Page 65: FOUND - ['abc']...

Page 66: FOUND - ['abc', 'abcd']...

Page 67: FOUND - ['abc', 'abcd', '//']...

Page 68: FOUND - ['abc']...

Page 70: FOUND -